# PathSnap model run: Deep MLP

This notebook trains only **Deep MLP** using trip-grouped cross-validation on the 80 training trips and compares it on the 10 student-test trips. It uses the shared PathSnap feature and evaluation code. It does not access evaluator data.

Use a high-RAM CPU runtime. The estimator is CPU based; a GPU is not required. Keep Google Drive mounted so the candidate cache and result files persist between notebooks.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/PathSnap')
if not (ROOT / 'scripts/run_experiments.py').exists():
    print('Upload PathSnap_source_bundle.zip, created by scripts/package_source.py')
    uploaded = files.upload()
    bundle = next((Path(name) for name in uploaded if name.lower().endswith('.zip')), None)
    if bundle is None:
        raise FileNotFoundError('Upload the PathSnap source bundle ZIP.')
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(ROOT)

ARCHIVE = ROOT / 'Student folder.zip'
if not ARCHIVE.exists():
    print('Upload the supplied Student folder.zip (do not upload evaluator data).')
    uploaded = files.upload()
    chosen = next((Path(name) for name in uploaded if name.lower().endswith('.zip')), None)
    if chosen is None:
        raise FileNotFoundError('Upload Student folder.zip.')
    shutil.copy2(chosen, ARCHIVE)

print('Project:', ROOT)
print('Student archive:', ARCHIVE)


In [ ]:
%cd /content/drive/MyDrive/PathSnap
%pip install -q -r requirements.txt


## Run the selected model

The first notebook prepares and saves `results/candidate_cache.joblib` on Drive. Later model notebooks reuse it. Let the preparation step finish; it processes each complete trip and does not split trips across folds.

In [ ]:
import json, subprocess, sys

config = json.loads((ROOT / 'configs/default.json').read_text(encoding='utf-8'))
config['data_archive'] = str(ARCHIVE)
config_path = Path('/content/pathsnap_mlp_deep_config.json')
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
output_dir = ROOT / 'results' / 'cloud_mlp_deep'
cache_path = ROOT / 'results' / 'candidate_cache.joblib'
command = [
    sys.executable, str(ROOT / 'scripts/run_experiments.py'),
    '--config', str(config_path), '--model', 'mlp_deep',
    '--cache-path', str(cache_path), '--output-dir', str(output_dir),
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=ROOT, check=True)


In [ ]:
import pandas as pd

result = json.loads((output_dir / 'experiment_results.json').read_text(encoding='utf-8'))
model_result = result['models']['mlp_deep']
print('Feature preparation seconds:', round(result['preparation_seconds'], 1))
print('Configuration:', model_result['configuration'])
display(pd.DataFrame(model_result['cv']['folds']))
display(pd.DataFrame([model_result['student_test']]))
print('Saved outputs:', output_dir)


## Return these run artifacts

Save this executed notebook with its cell outputs. Also share the matching `results/cloud_MODEL/` folder, which contains machine-readable fold metrics, student-test metrics and per-trip candidate predictions. Do not edit reported values by hand. The team will compare all eight run outputs before selecting and tuning a model.